In [ ]:
#| hide
#| eval: false
from dotenv import load_dotenv
load_dotenv('../.env')

True

In [ ]:
%load_ext autoreload
%autoreload 2

# utils

In [ ]:
#| default_exp: utils

In [ ]:
#| export
from IPython import get_ipython

We would like to add some cells programmatically. There seem to be some limitations given by the design of Jupyter; basically the file on disk, the kernel, and the frontend are different things. The following "payload" approach seems to be the only way to create a new cell without writing extensions for different frontends. However, this approach seems to be at risk of deprecation.

In [ ]:
#| export
def add_cell(content: str, single=False):
    """Add a new cell below the calling cell once overall execution of the cell finishes.
    
    Parameters
    ----------
    single : int
        If True, overwrites (updates) newly created cell when called multiple times.
    """
    if not isinstance(content, str): raise TypeError("Content should be string")
    if len(content) == 0: return
    shell = get_ipython()
    shell.payload_manager.write_payload(
        {"source":"set_next_input",
         "text":content,
         "replace": False}, # replaces the calling cell if True, I have never used this so it's not exposed
        single=single)

In [ ]:
add_cell("# Hi")

In [ ]:
# Hi

In [ ]:
# with single = True, the new cell will be updated (overwritten)
add_cell("# first", single=True)
add_cell("# second", single=True)

In [ ]:
# second

In [ ]:
#| export
def add_cells(contents: list[str]):
    """Adds several cells below the calling cells once excution of the current cell finishes
    
    Parameters
    ----------
    contents : list[str]
        The contents to be placed in the new cells.
    """
    if not isinstance(contents, list): raise TypeError("Contents should be list (of strings)")
    for c in reversed(contents): # the calls are scheduled and executed in reversed order
        add_cell(c, single=False)

In [ ]:
add_cells(["# Hi", "# there"])

In [ ]:
# Hi

In [ ]:
# there

The calls to the payload manager are scheduled and executed in order after the cell execution finishes.

In [ ]:
add_cell("# first")
add_cell("# second")

In [ ]:
# second

In [ ]:
# first

This means if you don't reverse the desired output order, cells will be in the wrong order. The last cell needs to be added (scheduled) first. This makes streaming impossible - you need to know the end first. You can only post-process fully received output.

Let's post-process markdown strings - add one cell per header (up to a certain level). This is helpful to split up long texts in small logical chunks, e.g., when importing a paper to Jupyter.

In [ ]:
#| export
def split_markdown(md: str, max_level: int = 6) -> list[str]:
    """Splits a markdown formatted string at headers"""
    try: from mdsplit import split_by_heading
    except ModuleNotFoundError: raise ModuleNotFoundError("This function requires mdsplit, https://github.com/markusstraub/mdsplit. Please run `pip install mdsplit` to use.")

    lines = md.splitlines(keepends=True) # keepends=True preserves original newline characters without alteration
    
    return ["".join(chapter.text) for chapter in split_by_heading(lines, max_level=max_level)]

Let's assign a `long_string` to demonstrate a use case:

In [ ]:
long_string = """# Cell 1
This is the content for the first cell.

## Section 2

If `max_level` is larger than 1, this section will be its own cell.

# Final cell
A last top-level cell with `# some comment`.
"""

In [ ]:
assert len(split_markdown(long_string, max_level=2)) == 3

The long document can then be added as individual cells like this:

In [ ]:
#| eval: false
add_cells(split_markdown(long_string, max_level=4))

# Cell 1
This is the content for the first cell.



## Section 2

If `max_level` is larger than 1, this section will be its own cell.



# Final cell
A last top-level cell with `# some comment`.


The resulting cells are unfortunately code cells - you need to go manually through them to convert them to markdown (`m`) and execute (`Shift + Enter`, execute and go to next cell). I currently don't have a good idea to make this automatic because as I understand this would only be possible from the frontend and therefore would require specialized code for each different frontend.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()